# Essence Wars: Agent Evaluation

This notebook demonstrates how to evaluate agents against built-in bots.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/essence-wars/blob/main/python/notebooks/03_evaluation.ipynb)

## Setup

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install essence-wars[train]

import numpy as np
import torch
from essence_wars import PyGame
from essence_wars._core import STATE_TENSOR_SIZE, ACTION_SPACE_SIZE

print(f"Observation size: {STATE_TENSOR_SIZE}")
print(f"Action space size: {ACTION_SPACE_SIZE}")

## Evaluation Helper Function

In [ ]:
def evaluate_matchup(agent_fn, opponent_fn, num_games=50, 
                     deck1='artificer_tokens', deck2='broodmother_pack'):
    """Evaluate an agent against an opponent.
    
    Args:
        agent_fn: Function taking PyGame and returning action (for player 0)
        opponent_fn: Function taking PyGame and returning action (for player 1)
        num_games: Number of games to play
        deck1: Agent's deck
        deck2: Opponent's deck
    
    Returns:
        dict with results
    """
    wins, losses, draws = 0, 0, 0
    
    for seed in range(num_games):
        game = PyGame(deck1=deck1, deck2=deck2)
        game.reset(seed=seed)
        
        while not game.is_done():
            if game.current_player() == 0:
                action = agent_fn(game)
            else:
                action = opponent_fn(game)
            game.step(action)
        
        reward = game.get_reward(0)
        if reward > 0:
            wins += 1
        elif reward < 0:
            losses += 1
        else:
            draws += 1
    
    return {
        'win_rate': wins / num_games,
        'wins': wins,
        'losses': losses,
        'draws': draws,
        'games': num_games,
    }

print("Evaluation function defined")

## Bot Definitions

In [ ]:
# Define bot functions
def random_bot(game):
    return game.random_action()

def greedy_bot(game):
    return game.greedy_action()

def mcts_50_bot(game):
    return game.mcts_action(50)

def mcts_100_bot(game):
    return game.mcts_action(100)

print("Bot functions defined: random, greedy, mcts_50, mcts_100")

## Baseline Comparisons

Establish baseline performance levels between built-in bots.

In [ ]:
print("Baseline Comparisons (30 games each)")
print("=" * 50)

# Random vs Greedy
result = evaluate_matchup(random_bot, greedy_bot, num_games=30)
print(f"Random vs Greedy:   {result['win_rate']*100:5.1f}% ({result['wins']}/{result['games']})")

# Greedy vs Random
result = evaluate_matchup(greedy_bot, random_bot, num_games=30)
print(f"Greedy vs Random:   {result['win_rate']*100:5.1f}% ({result['wins']}/{result['games']})")

# Greedy vs Greedy (deck effect)
result = evaluate_matchup(greedy_bot, greedy_bot, num_games=30)
print(f"Greedy vs Greedy:   {result['win_rate']*100:5.1f}% ({result['wins']}/{result['games']})")

## Greedy vs MCTS at Various Strengths

In [ ]:
print("\nGreedy vs MCTS at various simulation counts")
print("=" * 50)

# Greedy vs MCTS-50 (10 games for speed)
result = evaluate_matchup(greedy_bot, mcts_50_bot, num_games=10)
print(f"Greedy vs MCTS-50:  {result['win_rate']*100:5.1f}%")

# Greedy vs MCTS-100 (10 games for speed)
result = evaluate_matchup(greedy_bot, mcts_100_bot, num_games=10)
print(f"Greedy vs MCTS-100: {result['win_rate']*100:5.1f}%")

## Deck Matchup Analysis

In [ ]:
# Get available decks
all_decks = PyGame.list_decks()
test_decks = all_decks[:4]  # Test first 4 for speed

print("\nDeck Matchup Analysis (Greedy vs Greedy, 10 games)")
print("=" * 60)

for d1 in test_decks[:2]:  # Limit for speed
    for d2 in test_decks[2:]:
        result = evaluate_matchup(greedy_bot, greedy_bot, 
                                  num_games=10, deck1=d1, deck2=d2)
        print(f"{d1:20s} vs {d2:20s}: {result['win_rate']*100:5.1f}%")

## Evaluate a Neural Agent (Optional)

If you have a trained model, load and evaluate it here.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleNetwork(nn.Module):
    """Simple policy network."""
    def __init__(self, hidden_dim=128):
        super().__init__()
        self.fc = nn.Linear(STATE_TENSOR_SIZE, hidden_dim)
        self.policy = nn.Linear(hidden_dim, ACTION_SPACE_SIZE)
    
    def forward(self, obs, mask):
        x = F.relu(self.fc(obs))
        logits = self.policy(x)
        logits = torch.where(mask.bool(), logits, torch.tensor(-1e8))
        return logits

def make_neural_agent(network):
    """Create agent function from neural network."""
    def agent_fn(game):
        obs = torch.from_numpy(game.observe()).float().unsqueeze(0)
        mask = torch.from_numpy(game.action_mask()).float().unsqueeze(0)
        with torch.no_grad():
            logits = network(obs, mask)
            action = logits.argmax(dim=-1).item()
        return action
    return agent_fn

# Example: Create and evaluate a random-initialized network
network = SimpleNetwork()
neural_agent = make_neural_agent(network)

print("\nUntrained Neural Network vs Baselines")
print("=" * 50)

result = evaluate_matchup(neural_agent, random_bot, num_games=20)
print(f"Neural vs Random: {result['win_rate']*100:5.1f}%")

result = evaluate_matchup(neural_agent, greedy_bot, num_games=20)
print(f"Neural vs Greedy: {result['win_rate']*100:5.1f}%")

## Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Run evaluations
opponents = ['Random', 'Greedy']
greedy_results = []

for opp_fn, name in [(random_bot, 'Random'), (greedy_bot, 'Greedy')]:
    result = evaluate_matchup(greedy_bot, opp_fn, num_games=30)
    greedy_results.append(result['win_rate'] * 100)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(opponents, greedy_results, color=['#2ecc71', '#3498db'])

for bar, val in zip(bars, greedy_results):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
            f'{val:.0f}%', ha='center', fontsize=12)

ax.axhline(50, color='gray', linestyle='--', alpha=0.5, label='50% baseline')
ax.set_ylabel('Win Rate (%)')
ax.set_title('Greedy Bot Performance vs Baselines')
ax.set_ylim(0, 100)
ax.legend()
plt.tight_layout()
plt.show()

## Summary

Key takeaways:

1. **Random vs Greedy**: Greedy wins ~90%+ against random
2. **Greedy vs Greedy**: ~50% (measures deck balance)
3. **MCTS strength**: Higher simulations = stronger play
4. **Deck effects**: Some decks have advantages in certain matchups

For comprehensive evaluation, use the CLI:
```bash
essence-wars benchmark --checkpoint model.pt
```